# Thermodynamics and Entropy Diagnostics (software-test fixture only)

This notebook exercises configured workflow plumbing for explicit reaction-quotient Gibbs and entropy diagnostics. It is not a researcher-facing scientific example, empirical validation, calibration, or literature comparison, and its outputs must not be cited as biological evidence.

The configured validator uses caller-supplied dimensionless Q, temperature, and standard Gibbs metadata only. It does not infer activities, reaction quotients, redox potentials, or enforce thermodynamics during solver time.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import run_configured_model

SOURCE_CONFIG = ROOT / "data" / "model_configs" / "toy_homogeneous_ab.yml"
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "notebooks" / "examples" / "Outputs")))
OUTPUT = OUTPUT_ROOT / "11_thermodynamics_entropy_diagnostics"
CONFIG = OUTPUT_ROOT / "configured_inputs" / "explicit_q_thermodynamics.yml"

Create a tiny configured model from the existing toy benchmark and add one explicit-Q thermodynamic validator. The metadata values are synthetic software-test fixtures, not scientific measurements.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG.parent.mkdir(parents=True, exist_ok=True)

source = "Configured synthetic explicit-Q thermodynamics fixture; no scientific claim."
config = yaml.safe_load(SOURCE_CONFIG.read_text(encoding="utf-8"))
config["name"] = "explicit Q thermodynamics diagnostics benchmark"
config["mode"] = "toy"
config["maturity"] = "framework_benchmark"
config.setdefault("validators", []).append(
    {
        "id": "explicit_q_gibbs",
        "validator_type": "reaction_quotient_thermodynamic_metadata",
        "estimate": {
            "reaction_name": "configured synthetic condition-specific reaction",
            "source": source,
            "delta_gibbs": {
                "name": "configured standard delta G",
                "symbol": "dG_standard_configured",
                "value": -5.0,
                "units": "kilojoule / mole",
                "uncertainty": 0.0,
                "source": source,
                "confidence_level": "medium",
                "notes": "Synthetic standard Gibbs value for configured-output diagnostics only.",
                "measurement_method": "defined fixture value",
                "validity_range": "not a biological validity range",
            },
            "conditions": {
                "parameters": [
                    {
                        "name": "configured thermodynamic temperature",
                        "symbol": "T_configured",
                        "value": 298.15,
                        "units": "kelvin",
                        "uncertainty": 0.0,
                        "source": source,
                        "confidence_level": "medium",
                        "notes": "Synthetic condition value for configured-output diagnostics only.",
                        "measurement_method": "defined fixture value",
                        "validity_range": "not a biological validity range",
                    }
                ]
            },
        },
        "reaction_quotient": {
            "name": "configured dimensionless reaction quotient",
            "symbol": "Q_configured",
            "value": 1.0,
            "units": "dimensionless",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied Q for configured-output diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
        "temperature": {
            "name": "configured dynamic thermodynamic temperature",
            "symbol": "T_configured_dynamic",
            "value": 298.15,
            "units": "kelvin",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied temperature for configured-output diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
    }
)

CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
result = run_configured_model(CONFIG, output_dir=OUTPUT)
thermodynamic_rows = [row for row in result.validation_report() if row["name"] == "reaction_quotient_thermodynamic_feasibility"]

{
    "validation_status": thermodynamic_rows[0]["status"],
    "output_directory": str(OUTPUT),
}


Inspect the emitted JSON and CSV summaries. Both files are configured-run outputs generated by the package, not calculations implemented in the notebook.

In [ ]:
summary = json.loads((OUTPUT / "thermodynamic_summary.json").read_text(encoding="utf-8"))
with (OUTPUT / "thermodynamic_summary.csv").open(newline="", encoding="utf-8") as handle:
    csv_rows = list(csv.DictReader(handle))
manifest = json.loads((OUTPUT / "output_manifest.json").read_text(encoding="utf-8"))

row = summary["rows"][0]
{
    "summary_kind": summary["kind"],
    "summary_count": summary["count"],
    "gibbs_equation": row["gibbs_equation"],
    "entropy_equation": row["entropy_equation"],
    "csv_matches_json": csv_rows[0]["gibbs_equation"] == row["gibbs_equation"],
    "has_solver_time_enforcement": summary["has_solver_time_enforcement"],
    "summary_files": [name for name in manifest["files"] if name.startswith("thermodynamic_summary")],
    "unsupported_scope": summary["unsupported_scope"],
}
